# Location Selection with E-NAUTILUS: Part 2
_Running of E-NAUTILUS for decision making, and presentation of results_


In [14]:
import numpy as np
import pandas as pd
import polars as pl
import pickle
import folium
# These are to just suppress warnings in the outputs of the example
import warnings

warnings.filterwarnings("ignore")

## Load results from previous session

In [15]:
file_name = "data/pf_test.pkl"

output = open(file_name, 'rb')
prev_session = pickle.load(output)

raw_ref_pf = prev_session["pf"]
prob = prev_session["prob"]
sites = prev_session["sites"]
cities = prev_session["cities"]
cities_adj2sites = prev_session["cities_adj2sites"]


## Load reference front and problem 

In [16]:
output_flat = np.array(raw_ref_pf).flatten()

def process_lists(dict2conv):
    return {key: np.array(dict2conv[key]).flatten().tolist() for key in dict2conv.keys()}

# TODO include constraints in here too
output_dict = [
    output.optimal_objectives | 
    process_lists(output.optimal_variables) 
    for output in output_flat]

nd_df = pl.DataFrame(output_dict)

nd_df = nd_df.unique(subset=("f_1", "f_2", "f_3", "f_4"))

nd_df = nd_df.with_columns([
    (-pl.col("f_1")).alias("f_1_min"),
    (pl.col("f_2")).alias("f_2_min"),
    (pl.col("f_3")).alias("f_3_min"),
    (-pl.col("f_4")).alias("f_4_min")
])

nadir_point = {
  "f_1": float(nd_df["f_1"].min()),
  "f_2": float(nd_df["f_2"].max()),
  "f_3": float(nd_df["f_3"].max()),
  "f_4": float(nd_df["f_4"].min())
}

display(nd_df)

print(f"Nadir point: {nadir_point}")
print(f"Nadir point (problem): {prob.get_nadir_point()}")
print(f"Idedal point (problem): {prob.get_ideal_point()}")


reachable_indices = list(range(len(nd_df)))  # everything reachable from nadir


f_1,f_2,f_3,f_4,ev,cover,_alpha,f_1_min,f_2_min,f_3_min,f_4_min
f64,f64,f64,f64,list[f64],list[f64],list[f64],f64,f64,f64,f64
513.0,7.0,1428.435,0.991157,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[442.195992],-513.0,7.0,1428.435,-0.991157
372.0,1.0,680.6,0.747887,"[0.0, 0.0, … 0.0]","[0.0, 0.0, … 0.0]",[-0.247887],-372.0,1.0,680.6,-0.747887
548.0,4.0,1434.23,0.951322,"[1.0, 0.0, … 1.0]","[1.0, 1.0, … 0.0]",[0.16076],-548.0,4.0,1434.23,-0.951322
286.0,2.0,565.82,0.685985,"[0.0, 0.0, … 0.0]","[1.0, 0.0, … 0.0]",[-0.14601],-286.0,2.0,565.82,-0.685985


Nadir point: {'f_1': 286.0, 'f_2': 7.0, 'f_3': 1434.23, 'f_4': 0.6859850318352757}
Nadir point (problem): {'f_1': 0, 'f_2': 20, 'f_3': 2968.9900000000002, 'f_4': 0}
Idedal point (problem): {'f_1': 589, 'f_2': 0, 'f_3': 0, 'f_4': 1.0}


## Helper functions

In [17]:
# Function to determine marker size based on population
def get_marker_size(population):
    return max(5, population / 1000)  # Adjust the divisor to scale marker size

def create_color_dict(cities, ev_cities, cc): 
    marker_color = {}
    for city in cities.loc[:,"city"]: 
        if city in ev_cities: 
            marker_color[city] = "orange"
        elif city in cc: 
            marker_color[city] = "yellow"
        else: 
            marker_color[city] = "grey"

    return marker_color

def select_point(results, sol_id): 

    return {
        "f_1": int(results.loc[sol_id, "Total patients served"]),
        "f_2": int(results.loc[sol_id, "Number of under-attended sites"]),
        "f_3": float(results.loc[sol_id, "Total costs ($)"]),
        "f_4": float(results.loc[sol_id, "Population with access (%)"]/100.0)
        }


def clean_results(raw_results, intermediate_point=True): 
    # Transform objectives
    if intermediate_point: 
        results = pd.DataFrame(raw_results.intermediate_points)
    else:
        results = pd.DataFrame(raw_results.optimal_objectives)

    results = results.rename(columns={
                    "f_1": "Total patients served", 
                    "f_2": "Number of under-attended sites", 
                    "f_3": "Total costs ($)", 
                    "f_4": "Population with access (%)"})
    results[["Total patients served", "Number of under-attended sites"]] =  results[["Total patients served", "Number of under-attended sites"]].astype(int)
    results[["Population with access (%)"]] = (results[["Population with access (%)"]]*100.0).round(2)
    results[["Total costs ($)"]] = (results[["Total costs ($)"]]).round(2)

    results.index.name = "Solution ID"

    return results

## Run eNAUTILUS 
### Round 1
We're going to generate some solutions. They will be poor at first, but you and the computer will slowly find the best solution that fulfills your goals and preferences. 


In [18]:
# Initialize a first solution 
from desdeo.mcdm.enautilus import enautilus_step
from desdeo.mcdm.enautilus import enautilus_get_representative_solutions

current_iter = 0
selected_point = nadir_point
total_iterations = 3
display(f"Starting with point {selected_point}")

prob.get_ideal_point()


raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you prefer?")
display(results)

"Starting with point {'f_1': 286.0, 'f_2': 7.0, 'f_3': 1434.23, 'f_4': 0.6859850318352757}"

number of iterations left: 3


'Which solution to do you prefer?'

,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%)
Solution ID,,,,
0,361,6,1432.30,78.77
1,314,4,1183.02,70.66
2,286,5,1144.76,68.60


### Round 2 

In [19]:
chosen_solution = 0

In [20]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")
display(raw_results)
results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)
display("Results:")



{'f_1': 361, 'f_2': 6, 'f_3': 1432.3, 'f_4': 0.7877}
number of iterations left: 2


ENautilusResult(current_iteration=2, iterations_left=1, intermediate_points=[{'f_1': 437.0, 'f_2': 6.5, 'f_3': 1430.3674999999998, 'f_4': 0.88942843392784}, {'f_1': 366.5, 'f_2': 3.5, 'f_3': 1056.45, 'f_4': 0.7677934784227576}, {'f_1': 323.5, 'f_2': 4.0, 'f_3': 999.06, 'f_4': 0.7368425159176378}], reachable_best_bounds=[{'f_1': -inf, 'f_2': 7.0, 'f_3': 1434.23, 'f_4': -inf}, {'f_1': -inf, 'f_2': inf, 'f_3': inf, 'f_4': 0.7478869568455151}, {'f_1': 372.0, 'f_2': 1.0, 'f_3': 680.6, 'f_4': 0.7478869568455151}], reachable_worst_bounds=[{'f_1': 437.0, 'f_2': 6.5, 'f_3': 1430.3674999999998, 'f_4': 0.88942843392784}, {'f_1': 366.5, 'f_2': 3.5, 'f_3': 1056.45, 'f_4': 0.7677934784227576}, {'f_1': 323.5, 'f_2': 4.0, 'f_3': 999.06, 'f_4': 0.7368425159176378}], closeness_measures=[66.5202748446384, 50.923481111413935, 50.297189697743406], reachable_point_indices=[[], [], []])

'Which solution to do you find most preferable?'

,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%)
Solution ID,,,,
0,437,6,1430.37,88.94
1,366,3,1056.45,76.78
2,323,4,999.06,73.68


'Results:'

### Round 3

In [21]:
chosen_solution = 2

In [22]:
# Refine solution based on feedback
current_iter += 1
selected_point = select_point(results, chosen_solution)

print(selected_point)

raw_results = enautilus_step(
    problem=prob,
    non_dominated_points=nd_df,
    current_iteration=current_iter,
    iterations_left=total_iterations - current_iter,
    selected_point=selected_point,
    reachable_point_indices=reachable_indices,
    total_number_of_iterations=total_iterations,
    number_of_intermediate_points=3,
)

print(f"number of iterations left: {total_iterations - current_iter}")

results = clean_results(raw_results)
display("Which solution to do you find most preferable?")
display(results)




{'f_1': 323, 'f_2': 4, 'f_3': 999.06, 'f_4': 0.7368000000000001}
number of iterations left: 1


'Which solution to do you find most preferable?'

,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%)
Solution ID,,,,
0,513,7,1428.44,99.12
1,372,1,680.60,74.79
2,286,2,565.82,68.60


## Display final result

In [23]:
final_chosen_solution = 2

In [24]:
# Get final solution 
solutions = enautilus_get_representative_solutions(prob, raw_results, nd_df) 
solution = solutions[final_chosen_solution]
results = clean_results(solution, intermediate_point=False)

display(results)
 


,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%)
Solution ID,,,,
0,286,2,565.82,68.6


### Result postprocessing

In [25]:
# Post process result...
raw_sites = solution.optimal_variables['ev'][0].to_list()
raw_sites = [[bool(e) for e in raw_sites]]

raw_coverage = solution.optimal_variables['cover'][0].to_list()
raw_coverage = [[bool(c) for c in raw_coverage]]

sites_visited = []
for evb in raw_sites: 
    sites_visited.append("\n".join(sites.loc[evb, "site_id"].values))

cities_covered = [] 
for cc in raw_coverage: 
    cities_covered.append("\n".join(cities.loc[cc,"city"].values))

results["Sites Visited"] = sites_visited
results["Cities covered"] = cities_covered

results

,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,286,2,565.82,68.6,cairo-public-library\ndunkirk-hn-community-cen...,Ada\nBluffton\nCairo\nCaledonia\nColumbus Grov...


### Map preprocessing

In [26]:
# Process for the map
sites_in_cities = sites.loc[raw_sites[0],:].groupby("city").agg({"site_pretty": lambda e : '<br>'.join(e)})
sites_in_cities = sites_in_cities.to_dict()['site_pretty']
sites_in_cities

# cc cities covered
cc = set(cities_covered[0].split('\n'))

site_cities = list(sites.loc[raw_sites[0], "city"])
marker_colors = create_color_dict(cities, site_cities, cc)


## Site coverage dictionary 
site2city_mat = cities_adj2sites[raw_sites[0]].astype(bool)

site2city_dict = {}
for (c,city) in enumerate(site_cities): 
    site2city_dict[city] = set(cities.loc[site2city_mat[c,],"city"]) - {city}

adjacent_sites = {}
# Record what sites are near other cities
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_name = cities.loc[cities.loc[:,"city"] == site_city,["city"]].values.tolist()[0][0]

    for adj_city in adj_cities: 
        to_name = cities.loc[cities.loc[:,"city"] == adj_city ,["city"]].values.tolist()[0][0]

        if to_name not in adjacent_sites.keys(): 
            adjacent_sites[to_name] = {from_name}
        else: 
            adjacent_sites[to_name] = adjacent_sites[to_name].union({from_name})



## Render map

In [27]:
# Render map

# Create a base map
m = folium.Map(location=[cities['lat'].mean(), 
                         cities['long'].mean()], 
                         zoom_start=7) 

# Draw lines between 
for site_city in site2city_dict.keys(): 
    adj_cities = site2city_dict[site_city]
    from_loc = cities.loc[cities.loc[:,"city"] == site_city,["lat", "long"]].values.tolist()

    for adj_city in adj_cities: 
        to_loc = cities.loc[cities.loc[:,"city"] == adj_city ,["lat", "long"]].values.tolist()
        folium.PolyLine(
            locations=[to_loc[0], from_loc[0]],
            color="black"
        ).add_to(m)

# Set bounds
sw = cities.loc[:,['lat', 'long']].min().values.tolist()
ne = cities.loc[:,['lat', 'long']].max().values.tolist()
m.fit_bounds([sw,ne])

# Create tool tips 
tooltips = {}
for _, row in cities.iterrows():
    city = row['city']
    tooltips[city]=f"<b>{city}</b><br><b>Population:</b> {row['pop']}"

    if city in sites_in_cities.keys():
        tooltips[city]+= "<br><b>Sites with events:</b><br>"
        tooltips[city]+= sites_in_cities[city]
    else:
        tooltips[city]+= "<br><b>No Healthwise Clinics</b>"

    if city in adjacent_sites.keys(): 
        tooltips[city]+= "<br><b>Covered by events in: </b>"
        tooltips[city]+= "<br>".join(adjacent_sites[city])


# Add cities to the map
for _, row in cities.iterrows():
    city = row['city']
    folium.CircleMarker(
        location=(row['lat'], row['long']),
        radius=get_marker_size(row['pop']),
        color="black",
        fill=True,
        fill_color=marker_colors[city],
        fill_opacity=0.6,
        tooltip=tooltips[city]
    ).add_to(m)

display(results)
display(m)

,Total patients served,Number of under-attended sites,Total costs ($),Population with access (%),Sites Visited,Cities covered
Solution ID,,,,,,
0,286,2,565.82,68.6,cairo-public-library\ndunkirk-hn-community-cen...,Ada\nBluffton\nCairo\nCaledonia\nColumbus Grov...
